[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/abpandey-dev/learn-python-from-scratch-to-advanced/blob/main/notebooks/02-data-science/04_pandas-cleaning.ipynb)

*Python for Data Science · Notebook 4 of 8*

# Data Cleaning with Pandas

Missing values, duplicates, wrong types, messy text, dates and outliers — getting raw data ready for analysis.

### 👩‍🏫 From your teacher

**The picture to keep in your head:** Real data arrives like a messy room. Cleaning means deciding what to keep, fix or throw out, and writing down why.

**What interviewers really test here:** Missing values, duplicates, dtypes, outliers and not leaking information.

### How to use this notebook

1. **Run the setup cell** just below.
2. **Read each lesson slowly.** Work the numbers on paper before reading on.
3. **Run and then change the examples.** Breaking code on purpose is how you learn.
4. **Try each practice question before opening an answer.** If you're stuck, open one hint at a time.
5. **Say your interview answers out loud.** Knowing something and explaining it are different skills.

### Your roadmap

- [ ] 1. Why clean data? *(Basics)*
- [ ] 2. How missing values really behave *(Basics)*
- [ ] 3. Missing values *(Core)*
- [ ] 4. Types and duplicates *(Core)*
- [ ] 5. Cleaning text with .str *(Core)*
- [ ] 6. Choosing how to fill gaps, worked *(Core)*
- [ ] 7. Outliers: find them, then decide *(Core)*
- [ ] 8. Dates, categories and outliers *(Advanced)*
- [ ] 9. A complete cleaning walk-through *(Advanced)*
- [ ] 20 practice questions
- [ ] Interview questions, out loud

In [1]:
# ▶ Run this cell first. It prepares the notebook; you don't need to read it.
import contextlib, copy, io, math, sys
import numpy as np, matplotlib.pyplot as plt
%matplotlib inline
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
PLAIN = True

class ListNode:
    def __init__(self, val=0, next=None):
        self.val, self.next = val, next

class TreeNode:
    def __init__(self, val=0, left=None, right=None):
        self.val, self.left, self.right = val, left, right

def mk_list(vals, pos=-1):
    dummy = cur = ListNode()
    nodes = []
    for v in vals:
        cur.next = ListNode(v)
        cur = cur.next
        nodes.append(cur)
    if pos >= 0 and nodes:
        nodes[-1].next = nodes[pos]
    return dummy.next

def list_arr(head):
    out, seen = [], set()
    while head and id(head) not in seen:
        seen.add(id(head))
        out.append(head.val)
        head = head.next
    return out

def mk_tree(vals):
    if not vals or vals[0] is None:
        return None
    root = TreeNode(vals[0])
    queue, i = [root], 1
    while queue and i < len(vals):
        node = queue.pop(0)
        for side in ("left", "right"):
            if i < len(vals) and vals[i] is not None:
                child = TreeNode(vals[i])
                setattr(node, side, child)
                queue.append(child)
            i += 1
    return root

def tree_arr(root):
    out, queue = [], [root]
    while queue:
        node = queue.pop(0)
        if node:
            out.append(node.val)
            queue += [node.left, node.right]
        else:
            out.append(None)
    while out and out[-1] is None:
        out.pop()
    return out

def find(root, val):
    if not root or root.val == val:
        return root
    return find(root.left, val) or find(root.right, val)

ListNode.__repr__ = lambda self: f"ListNode{list_arr(self)}"
TreeNode.__repr__ = lambda self: f"TreeNode{tree_arr(self)}"

def plain(x, digits=4):
    """NumPy / pandas results as plain Python values, so they compare with a test's expected answer."""
    np = sys.modules.get("numpy")
    pd = sys.modules.get("pandas")
    if pd is not None:
        if isinstance(x, pd.DataFrame):
            return {str(c): plain(x[c].tolist(), digits) for c in x.columns}
        if isinstance(x, pd.Series):
            return {plain(k, digits): plain(v, digits) for k, v in x.items()}
        if isinstance(x, pd.Index):
            return plain(x.tolist(), digits)
        if x is pd.NaT:
            return None
        if isinstance(x, pd.Timestamp):
            return str(x.date()) if x == x.normalize() else str(x)
    if np is not None:
        if isinstance(x, np.ndarray):
            return plain(x.tolist(), digits)
        if isinstance(x, np.generic):
            return plain(x.item(), digits)
    if isinstance(x, float):
        return None if x != x else round(x, digits) + 0.0
    if isinstance(x, list):
        return [plain(v, digits) for v in x]
    if isinstance(x, tuple):
        return tuple(plain(v, digits) for v in x)
    if isinstance(x, dict):
        return {plain(k, digits): plain(v, digits) for k, v in x.items()}
    if isinstance(x, (set, frozenset)):
        return {plain(v, digits) for v in x}
    return x


def check(fn, tests, plain_compare=None, listify=False, design=False, prep=None, post=None,
          norm=None, verify=None, runner=None, inplace=None):
    """Run your code against the tests and show which pass."""
    use_plain = PLAIN if plain_compare is None else plain_compare
    passed, problems = 0, []
    for args, want in tests:
        try:
            a = copy.deepcopy(args)
            if design:
                obj = fn(*a[0])
                got = [getattr(obj, m)(*p) for m, p in a[1]]
            elif runner is not None:
                got = runner(fn, a)
            else:
                if prep is not None:
                    a = prep(a)
                with contextlib.redirect_stdout(io.StringIO()):
                    got = fn(*a)
                if inplace is not None:
                    got = a[inplace]
                if listify:
                    got = list(got)
                if post is not None:
                    got = post(got)
            if verify is not None:
                ok = bool(verify(copy.deepcopy(args), got))
            elif norm is not None:
                ok = norm(copy.deepcopy(got)) == norm(copy.deepcopy(want))
            else:
                ok = (plain(got), plain(want)) if use_plain else (got, want)
                ok = ok[0] == ok[1]
            ok = ok if isinstance(ok, bool) else False
        except Exception as e:  # your code raised an error: report it, don't stop the notebook
            ok, got = False, f"{type(e).__name__}: {e}"
        passed += ok
        if not ok:
            problems.append((args, want, got))
    plt.close("all")
    if passed == len(tests):
        print(f"✅ All {len(tests)} tests passed. Well done! Now compare with the answers below.")
    else:
        print(f"❌ {passed}/{len(tests)} tests passed. First one to fix:")
        args, want, got = problems[0]
        print("   input:   ", repr(args)[:300])
        print("   expected:", repr(want)[:300])
        print("   you got: ", repr(got)[:300])
    return None

print("Ready! Scroll down and start learning.")

Ready! Scroll down and start learning.


---
# Part 1: The theory in one place

Read this once for the big picture. It's fine if it doesn't all stick yet: the lessons will make it concrete, and you can come back to revise.

### Data cleaning (data wrangling)

**Data cleaning** is the process of detecting and correcting (or removing) inaccurate, incomplete, inconsistent, duplicated or irrelevant data so that analysis is valid. It is part of **data wrangling / data preparation**, which commonly takes the majority of an analyst's time.



#### Dimensions of data quality



| Dimension | Question | Example problem |
| --- | --- | --- |
| Completeness | Is anything missing? | blank ages |
| Validity | Does it follow the rules? | age = −4, email without @ |
| Accuracy | Is it correct? | typo in a salary |
| Consistency | Is it the same everywhere? | "Pune", "pune ", "PNQ" |
| Uniqueness | Any duplicates? | the same order twice |
| Timeliness | Is it current? | last year's prices |

### Missing data

Representations: `np.nan` (float), `None` (object), `pd.NA` (nullable dtypes), `pd.NaT` (datetimes). `isna()` detects all of them.



#### Why data is missing




- **MCAR** — missing completely at random (a sensor glitch): dropping rows is unbiased.
- **MAR** — missing at random given other columns (older people skip the income question): impute using those columns.
- **MNAR** — missing not at random (high earners hide income): any simple fix is biased; flag it.



#### Strategies



| Strategy | pandas | Trade-off |
| --- | --- | --- |
| Delete rows / columns | `dropna` | loses data; biased unless MCAR |
| Constant / "Unknown" | `fillna(value)` | simple; may create a fake category |
| Mean / median / mode | `fillna(s.mean())` | keeps rows; shrinks variance (median is robust to outliers) |
| Forward / backward fill | `ffill`, `bfill` | good for ordered/time data |
| Interpolation | `interpolate` | smooth series; assumes a trend |
| Model-based | scikit-learn imputers | most accurate; more work |



Adding an indicator column (`df["age_missing"] = df["age"].isna()`) keeps the information that a value was imputed.

### Types, duplicates and text

- **Type conversion**: `astype` (strict), `pd.to_numeric(errors="coerce")`, `pd.to_datetime(errors="coerce", format=…)`, `astype("category")`.
- **Duplicates**: `duplicated(subset=, keep="first"|"last"|False)`, `drop_duplicates`. Decide which columns define "the same record".
- **Text normalisation**: strip whitespace, unify case, remove punctuation/currency symbols, standardise spellings with `replace`/`map`.
- **Regular expressions** in `.str.replace/contains/extract/match`: `\d` digit, `\D` non-digit, `\s` whitespace, `+` one or more, `^ $` start/end.

### Dates and times

| Object | Meaning |
| --- | --- |
| `Timestamp` / `datetime64` | a moment in time |
| `Timedelta` | a duration (difference of two timestamps) |
| `Period` | a span such as a month (`to_period("M")`) |
| `DatetimeIndex` | an index of dates, enabling `resample` and date slicing |



The `.dt` accessor exposes parts: `year, month, day, dayofweek, day_name(), hour, quarter, is_month_end`. Time zones: `tz_localize`, `tz_convert`.

### Outliers

An **outlier** is a value far from the rest. It may be an error (typo, unit mix-up) or a genuine extreme (a celebrity's income).



| Detection rule | Outlier if |
| --- | --- |
| IQR (Tukey's fences) | x &lt; Q1 − 1.5·IQR or x &gt; Q3 + 1.5·IQR |
| Z-score | \|x − mean\| / std &gt; 3 |
| Domain rules | age &gt; 120, negative price |



Treatments: correct, remove, cap (**winsorise** with `clip`), transform (log), or keep and use robust statistics (median, IQR).

---
# Part 2: Lessons, step by step

## Lesson 1: Why clean data?

*Level: Basics*

Real data is messy: blank cells, the same customer entered twice, numbers stored as text, " Pune" and "pune" counted as different cities, dates in strange formats, one salary typed as 9,000,000 by mistake. Analysts often spend most of their time cleaning, because **wrong input gives confident-looking wrong answers**.



A cleaning checklist:




1. Look: `df.info()`, `df.describe()`, `df.isna().sum()`, `df.duplicated().sum()`.
2. Fix types (numbers, dates, categories).
3. Handle missing values.
4. Remove duplicates.
5. Standardise text.
6. Check ranges and outliers.

## Lesson 2: How missing values really behave

*Level: Basics*

Missing values are quietly treated differently from what you may expect. Take `s = pd.Series([1, None, 5])`:



| Code | Result | Why |
| --- | --- | --- |
| `s.sum()` | 6 | skips the gap |
| `s.mean()` | 3 | 6 / 2, not 6 / 3 |
| `s.count()` | 2 | counts present values |
| `len(s)` | 3 | counts all rows |
| `s == None` | all False | never test for gaps this way |
| `s.isna()` | False, True, False | the right way |
| the dtype | float64 | a gap turned the whole-number column to decimals |



So if ids suddenly look like 3.0, a gap is probably the reason. `convert_dtypes()` or the `Int64` type keeps them whole.



#### Ask why it's missing



- **Random:** a form glitch. Filling is fairly safe.
- **Linked to other columns:** younger people skip the income question. Fill within groups.
- **Linked to the value itself:** very high earners hide their income. Any fill will bias results, so keep a "was missing" flag.

**▶ See it run.** Before you run this, predict what will happen. Then change a number and run it again.

In [2]:
def missing_behaviour():
    s = pd.Series([1, None, 5])
    total = s.sum()
    average = s.mean()
    present = s.count()
    rows = len(s)
    wrong_test = (s == None).sum()
    right_test = s.isna().sum()
    kind = str(s.dtype)


missing_behaviour()

**✏️ Your turn:** you now have what you need for [Q1](#q1), [Q2](#q2). Try one now, or carry on and do them in Part 3.

## Lesson 3: Missing values

*Level: Core*

pandas marks missing data as `NaN` (or `None` / `pd.NA` / `NaT` for dates). Most calculations skip them automatically.



| Task | Code |
| --- | --- |
| Find | `df.isna()`, `df.isna().sum()`, `df[df["age"].isna()]` |
| Drop rows | `df.dropna()`, `df.dropna(subset=["age"])` |
| Drop columns | `df.dropna(axis=1, thresh=n)` |
| Fill with a value | `df["city"].fillna("Unknown")` |
| Fill with a statistic | `s.fillna(s.mean())`, `s.fillna(s.median())` |
| Fill from neighbours | `s.ffill()`, `s.bfill()`, `s.interpolate()` |



**⚠️ Watch out:** Dropping rows can bias results if the missing values aren't random. Filling with the mean shrinks the spread. Choose deliberately and say what you did.

**▶ See it run.** Before you run this, predict what will happen. Then change a number and run it again.

In [3]:
def missing_demo():
    df = pd.DataFrame({"name": ["Ana", "Ben", "Cy", "Dia"], "age": [21, None, 30, None], "city": ["Pune", None, "Goa", "Pune"]})
    missing = df.isna().sum()
    complete = df.dropna()
    df["age"] = df["age"].fillna(df["age"].mean())
    df["city"] = df["city"].fillna("Unknown")
    print(missing.to_dict(), len(complete))
    print(df)


missing_demo()

{'name': 0, 'age': 2, 'city': 1} 2
  name   age     city
0  Ana  21.0     Pune
1  Ben  25.5  Unknown
2   Cy  30.0      Goa
3  Dia  25.5     Pune


**✏️ Your turn:** you now have what you need for [Q1](#q1), [Q2](#q2), [Q3](#q3), [Q4](#q4), [Q11](#q11), [Q19](#q19), [Q20](#q20). Try one now, or carry on and do them in Part 3.

## Lesson 4: Types and duplicates

*Level: Core*

Numbers read as text can't be added (`"10" + "5"` is `"105"`). Fix types early:



```python
df["qty"] = df["qty"].astype(int)
df["price"] = pd.to_numeric(df["price"], errors="coerce")   # bad values → NaN
df["date"] = pd.to_datetime(df["date"])
df["city"] = df["city"].astype("category")                 # saves memory
```



Duplicates inflate counts and sums:



```python
df.duplicated().sum()                      # how many repeats
df.drop_duplicates()                       # keep the first occurrence
df.drop_duplicates(subset=["email"])       # same email = same person
```

**▶ See it run.** Before you run this, predict what will happen. Then change a number and run it again.

In [4]:
def types_demo():
    df = pd.DataFrame({"item": ["pen", "pen", "ink"], "qty": ["3", "3", "2"], "price": ["10", "10", "n/a"]})
    df["qty"] = df["qty"].astype(int)
    df["price"] = pd.to_numeric(df["price"], errors="coerce")
    repeats = df.duplicated().sum()
    df = df.drop_duplicates()
    print(df.dtypes.astype(str).to_dict(), repeats)
    print(df)


types_demo()

{'item': 'str', 'qty': 'int64', 'price': 'float64'} 1
  item  qty  price
0  pen    3   10.0
2  ink    2    NaN


**✏️ Your turn:** you now have what you need for [Q12](#q12), [Q17](#q17), [Q5](#q5). Try one now, or carry on and do them in Part 3.

## Lesson 5: Cleaning text with .str

*Level: Core*

The `.str` accessor applies string methods to a whole column:



| Task | Code |
| --- | --- |
| Trim and normalise case | `s.str.strip().str.lower()` |
| Replace | `s.str.replace(",", "")`; with regex: `s.str.replace(r"\D", "", regex=True)` |
| Split into columns | `s.str.split(" ", n=1, expand=True)` |
| Take a piece | `s.str[:3]`, `s.str.split("@").str[1]` |
| Test | `s.str.contains("pune", case=False)`, `s.str.match(regex)`, `s.str.startswith("A")` |
| Extract with a pattern | `s.str.extract(r"(\d+)")` |



Clean column names too: `df.columns = df.columns.str.strip().str.lower().str.replace(" ", "_")`.

**▶ See it run.** Before you run this, predict what will happen. Then change a number and run it again.

In [5]:
def text_demo():
    df = pd.DataFrame({"name": ["  ada LOVELACE", "Alan turing "], "price": ["₹1,200", "Rs 75"]})
    df["name"] = df["name"].str.strip().str.title()
    df["price"] = df["price"].str.replace(r"\D", "", regex=True).astype(int)
    parts = df["name"].str.split(" ", n=1, expand=True)
    df["first"] = parts[0]
    print(df)


text_demo()

           name  price first
0  Ada Lovelace   1200   Ada
1   Alan Turing     75  Alan


**✏️ Your turn:** you now have what you need for [Q6](#q6), [Q13](#q13), [Q14](#q14), [Q18](#q18), [Q10](#q10). Try one now, or carry on and do them in Part 3.

## Lesson 6: Choosing how to fill gaps, worked

*Level: Core*

| Situation | Choice |
| --- | --- |
| Very few rows affected | drop those rows |
| Numbers, roughly symmetric | mean |
| Numbers with outliers | median |
| Categories | most common, or "Unknown" |
| Time series | carry forward, or interpolate |
| Values depend on a group | the group's own average |
| Over 60–70% missing | consider dropping the column |



#### Worked example: group filling



| City | Rent | Global mean fill | Group mean fill |
| --- | --- | --- | --- |
| Pune | 10 | 10 | 10 |
| Pune | missing | 20 | **15** |
| Pune | 20 | 20 | 20 |
| Delhi | 30 | 30 | 30 |
| Delhi | missing | 20 | **30** |



The global mean (20) gives a Delhi flat a Pune-like price. The group mean respects the difference:



```python
df["rent"].fillna(df.groupby("city")["rent"].transform("mean"))
```



**Filling shrinks the spread,** since many rows get the same value, and can make relationships look different. Note how much you filled when you report results.

**▶ See it run.** Before you run this, predict what will happen. Then change a number and run it again.

In [6]:
def group_fill():
    df = pd.DataFrame({"city": ["Pune", "Pune", "Pune", "Delhi", "Delhi"], "rent": [10, None, 20, 30, None]})
    global_mean = df["rent"].mean()
    global_filled = df["rent"].fillna(global_mean)
    group_means = df.groupby("city")["rent"].transform("mean")
    group_filled = df["rent"].fillna(group_means)


group_fill()

**✏️ Your turn:** you now have what you need for [Q3](#q3), [Q4](#q4), [Q11](#q11), [Q19](#q19), [Q20](#q20). Try one now, or carry on and do them in Part 3.

## Lesson 7: Outliers: find them, then decide

*Level: Core*

#### Find them with the IQR rule



Values: 10, 12, 13, 14, 15, 16, 100.



1. Q1 = 12.5 and Q3 = 15.5, so IQR = 3.
2. Lower fence: 12.5 − 1.5 × 3 = 8. Upper fence: 15.5 + 1.5 × 3 = 20.
3. 100 is outside the fences, so it's flagged.



One value drags the mean from 13.33 up to **25.71**, while the median is just 14.



#### Then decide, don't just delete



| What it is | What to do |
| --- | --- |
| Data entry error (age 200) | fix it, or set it to missing |
| Wrong unit (height 170 among values in metres) | convert |
| Real but extreme (a billionaire's income) | keep it; use median or a log transform |
| Real and it distorts a model | cap it (`clip`) and say so |
| The thing you're looking for (fraud) | **definitely keep it** |



Always look at flagged rows yourself, and record what you removed and why.

**▶ See it run.** Before you run this, predict what will happen. Then change a number and run it again.

In [7]:
def outlier_steps():
    s = pd.Series([10, 12, 13, 14, 15, 16, 100])
    q1, q3 = s.quantile(0.25), s.quantile(0.75)
    iqr = q3 - q1
    low, high = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    flagged = s[(s < low) | (s > high)]
    mean_with, mean_without, median = s.mean(), s[s <= high].mean(), s.median()
    capped = s.clip(low, high)


outlier_steps()

**✏️ Your turn:** you now have what you need for [Q9](#q9). Try one now, or carry on and do them in Part 3.

## Lesson 8: Dates, categories and outliers

*Level: Advanced*

```python
d = pd.to_datetime(df["date"])
d.dt.year, d.dt.month, d.dt.day_name()          # parts of a date
(d2 - d1).dt.days                               # durations
df["band"] = pd.cut(df["age"], bins=[0, 17, 59, 120], labels=["child", "adult", "senior"])
df["code"].map({"M": "male", "F": "female"})    # recode
df["income"].clip(lower, upper)                 # cap outliers
```



Outliers: look at `describe()` and a box plot. Common rules mark values beyond 1.5 × IQR from the quartiles, or more than 3 standard deviations from the mean (see the Statistics topic). Decide whether they are errors (fix/drop) or real (keep, maybe cap).

**▶ See it run.** Before you run this, predict what will happen. Then change a number and run it again.

In [8]:
def dates_demo():
    df = pd.DataFrame({"start": ["2024-01-01", "2024-02-27"], "end": ["2024-01-31", "2024-03-01"], "age": [15, 64]})
    start = pd.to_datetime(df["start"])
    end = pd.to_datetime(df["end"])
    df["days"] = (end - start).dt.days
    df["month"] = start.dt.month_name()
    df["band"] = pd.cut(df["age"], bins=[0, 17, 59, 120], labels=["child", "adult", "senior"])
    print(df)


dates_demo()

        start         end  age  days     month    band
0  2024-01-01  2024-01-31   15    30   January   child
1  2024-02-27  2024-03-01   64     3  February  senior


**✏️ Your turn:** you now have what you need for [Q7](#q7), [Q15](#q15), [Q8](#q8), [Q16](#q16), [Q9](#q9). Try one now, or carry on and do them in Part 3.

## Lesson 9: A complete cleaning walk-through

*Level: Advanced*

A small, realistically messy table, cleaned in order. The demo runs each step.



1. **Column names:** lower case, no spaces.
2. **Duplicates:** remove repeated rows.
3. **Text:** strip spaces, fix capitals, so " pune" and "PUNE" match.
4. **Numbers stored as text:** remove "₹" and commas, then `pd.to_numeric(errors="coerce")` turns anything unreadable into a gap.
5. **Dates:** `pd.to_datetime(errors="coerce")`. Check whether 03/04 means March or April: `dayfirst=True` sets it.
6. **Gaps:** fill or drop, knowing why.
7. **Check the result:** shape, types, and `describe()` again.



**Why `errors="coerce"`?** It turns bad values into visible gaps you can count, rather than crashing or hiding them. But check how many appeared: a sudden jump means your format assumption was wrong.



Keep cleaning in code, never by editing the file by hand, so it can be rerun on next month's data.

**▶ See it run.** Before you run this, predict what will happen. Then change a number and run it again.

In [9]:
def clean_walkthrough():
    raw = pd.DataFrame({"Customer Name": [" ana ", "BEN", "BEN", "cy"], "City": [" pune", "Delhi", "Delhi", "PUNE "], "Price": ["₹1,200", "₹900", "₹900", "n/a"], "Joined": ["2024-01-15", "2024-02-20", "2024-02-20", "bad date"]})
    df = raw.rename(columns=lambda c: c.strip().lower().replace(" ", "_"))
    df = df.drop_duplicates()
    df["customer_name"] = df["customer_name"].str.strip().str.title()
    df["city"] = df["city"].str.strip().str.title()
    df["price"] = pd.to_numeric(df["price"].str.replace("₹", "").str.replace(",", ""), errors="coerce")
    df["joined"] = pd.to_datetime(df["joined"], errors="coerce")
    new_gaps = df.isna().sum()
    df["price"] = df["price"].fillna(df["price"].median())
    clean_shape = df.shape


clean_walkthrough()

**✏️ Your turn:** you now have what you need for [Q10](#q10), [Q5](#q5), [Q6](#q6), [Q13](#q13), [Q17](#q17), [Q12](#q12), [Q7](#q7). Try one now, or carry on and do them in Part 3.

---
# Part 3: Practice like an interview

There are 20 questions here, easiest first.

**Work the way you would in a real interview:** restate the problem, try an example by hand, say the simple approach and its cost, improve it, then write the code. Test it with the cell. Only open an answer once you've made a real attempt, because struggling is where the learning happens.

<a id="q1"></a>
### Q1. Count Missing Values

🟢 Warm-up · *isna().sum()*

Return {column: number of missing (None) values}.

**Examples**

- `data = {'a': [1, None, 3], 'b': [None, None, 'x']}` → `{'a': 1, 'b': 2}`

In [ ]:
def missing_counts(data):
    """
    Return {column: number of missing (None) values}.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(missing_counts, [
    (({'a': [1, None, 3], 'b': [None, None, 'x']},), {'a': 1, 'b': 2}),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Loop over each column and count the Nones.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

`df.isna().sum()` — a True/False table summed per column.

</details>

<details>
<summary><b>✅ Answer: Brute force — plain Python</b></summary>

**Idea:** Loop over each column and count the Nones.

```python
def missing_counts(data):
    out = {}
    for col, values in data.items():
        out[col] = sum(1 for v in values if v is None)
    return out
```

**Cost:** O(cells) time, O(cols) space

</details>

<details>
<summary><b>✅ Answer: Optimal — the library way</b></summary>

**Idea:** `df.isna().sum()` — a True/False table summed per column.

```python
def missing_counts(data):
    df = pd.DataFrame(data)
    return df.isna().sum()
```

**Cost:** O(cells) time, O(cells) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Loop over each column and count the Nones. That costs O(cells) time and O(cols) space."

"I can do better: `df.isna().sum()` — a True/False table summed per column. That's O(cells) time and O(cells) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/data-science/#/q/pandas-cleaning/count-missing-values)

<a id="q2"></a>
### Q2. Drop Incomplete Rows

🟢 Warm-up · *dropna*

Remove every row that has a missing value; return the remaining names.

**Examples**

- `data = {'name': ['Ana', 'Ben', 'Cy'], 'age': [21, None, 30], 'city': ['Pune', 'Goa', None]}` → `['Ana']`

In [ ]:
def complete_names(data):
    """
    Remove every row that has a missing value; return the remaining names.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(complete_names, [
    (({'name': ['Ana', 'Ben', 'Cy'], 'age': [21, None, 30], 'city': ['Pune', 'Goa', None]},), ['Ana']),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Keep row i only if no column has None at i.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

`df.dropna()`.

</details>

<details>
<summary><b>✅ Answer: Brute force — plain Python</b></summary>

**Idea:** Keep row i only if no column has None at i.

```python
def complete_names(data):
    n = len(data["name"])
    keep = []
    for i in range(n):
        if all(data[c][i] is not None for c in data):
            keep.append(data["name"][i])
    return keep
```

**Cost:** O(cells) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: Optimal — the library way</b></summary>

**Idea:** `df.dropna()`.

```python
def complete_names(data):
    return pd.DataFrame(data).dropna()["name"].tolist()
```

**Cost:** O(cells) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Keep row i only if no column has None at i. That costs O(cells) time and O(n) space."

"I can do better: `df.dropna()`. That's O(cells) time and O(n) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/data-science/#/q/pandas-cleaning/drop-incomplete-rows)

<a id="q3"></a>
### Q3. Fill Gaps With the Average

🟢 Warm-up · *fillna(mean)*

Replace missing ages with the average of the known ages (rounded to 1 decimal); return the ages.

**Examples**

- `data = {'age': [20, None, 30, None]}` → `[20.0, 25.0, 30.0, 25.0]`

In [ ]:
def fill_mean(data):
    """
    Replace missing ages with the average of the known ages (rounded to 1
    decimal); return the ages.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(fill_mean, [
    (({'age': [20, None, 30, None]},), [20.0, 25.0, 30.0, 25.0]),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Average the known values, then rebuild the list.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

`s.fillna(s.mean())` — mean skips NaN automatically.

</details>

<details>
<summary><b>✅ Answer: Brute force — plain Python</b></summary>

**Idea:** Average the known values, then rebuild the list.

```python
def fill_mean(data):
    known = [a for a in data["age"] if a is not None]
    avg = round(sum(known) / len(known), 1)
    return [float(a) if a is not None else avg for a in data["age"]]
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: Optimal — the library way</b></summary>

**Idea:** `s.fillna(s.mean())` — mean skips NaN automatically.

```python
def fill_mean(data):
    age = pd.DataFrame(data)["age"]
    return age.fillna(round(age.mean(), 1)).tolist()
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Average the known values, then rebuild the list. That costs O(n) time and O(n) space."

"I can do better: `s.fillna(s.mean())` — mean skips NaN automatically. That's O(n) time and O(n) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/data-science/#/q/pandas-cleaning/fill-gaps-with-the-average)

<a id="q4"></a>
### Q4. Fill Missing Text

🟢 Warm-up · *fillna(value)*

Replace missing cities with 'Unknown'.

**Examples**

- `data = {'city': ['Pune', None, 'Goa']}` → `['Pune', 'Unknown', 'Goa']`

In [ ]:
def fill_city(data):
    """
    Replace missing cities with 'Unknown'.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(fill_city, [
    (({'city': ['Pune', None, 'Goa']},), ['Pune', 'Unknown', 'Goa']),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

List comprehension with a condition.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

`df['city'].fillna('Unknown')`.

</details>

<details>
<summary><b>✅ Answer: Brute force — plain Python</b></summary>

**Idea:** List comprehension with a condition.

```python
def fill_city(data):
    return [c if c is not None else "Unknown" for c in data["city"]]
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: Optimal — the library way</b></summary>

**Idea:** `df['city'].fillna('Unknown')`.

```python
def fill_city(data):
    return pd.DataFrame(data)["city"].fillna("Unknown").tolist()
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: List comprehension with a condition. That costs O(n) time and O(n) space."

"I can do better: `df['city'].fillna('Unknown')`. That's O(n) time and O(n) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/data-science/#/q/pandas-cleaning/fill-missing-text)

<a id="q5"></a>
### Q5. Remove Duplicate Rows

🟢 Warm-up · *drop\_duplicates*

Remove rows that are exact repeats of an earlier row; return the remaining (name, city) pairs.

**Examples**

- `data = {'name': ['Ana', 'Ben', 'Ana', 'Ana'], 'city': ['Pune', 'Goa', 'Pune', 'Goa']}` → `[('Ana', 'Pune'), ('Ben', 'Goa'), ('Ana', 'Goa')]`

In [ ]:
def unique_rows(data):
    """
    Remove rows that are exact repeats of an earlier row; return the remaining
    (name, city) pairs.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(unique_rows, [
    (({'name': ['Ana', 'Ben', 'Ana', 'Ana'], 'city': ['Pune', 'Goa', 'Pune', 'Goa']},), [('Ana', 'Pune'), ('Ben', 'Goa'), ('Ana', 'Goa')]),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Keep a set of seen rows (as tuples).

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

`df.drop_duplicates()` (use `subset=` to compare only some columns).

</details>

<details>
<summary><b>✅ Answer: Brute force — plain Python</b></summary>

**Idea:** Keep a set of seen rows (as tuples).

```python
def unique_rows(data):
    seen, out = set(), []
    for row in zip(data["name"], data["city"]):
        if row not in seen:
            seen.add(row)
            out.append(row)
    return out
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: Optimal — the library way</b></summary>

**Idea:** `df.drop_duplicates()` (use `subset=` to compare only some columns).

```python
def unique_rows(data):
    df = pd.DataFrame(data).drop_duplicates()
    return list(df.itertuples(index=False, name=None))
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Keep a set of seen rows (as tuples). That costs O(n) time and O(n) space."

"I can do better: `df.drop_duplicates()` (use `subset=` to compare only some columns). That's O(n) time and O(n) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/data-science/#/q/pandas-cleaning/remove-duplicate-rows)

<a id="q6"></a>
### Q6. Tidy Up Text

🟢 Warm-up · *.str accessor*

Strip spaces and lower-case every name.

**Examples**

- `data = {'name': ['  Ana ', 'BEN', 'cY  ']}` → `['ana', 'ben', 'cy']`

In [ ]:
def tidy_names(data):
    """
    Strip spaces and lower-case every name.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(tidy_names, [
    (({'name': ['  Ana ', 'BEN', 'cY  ']},), ['ana', 'ben', 'cy']),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Call strip and lower on each string.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

The `.str` accessor applies string methods to the whole column: `.str.strip().str.lower()`.

</details>

<details>
<summary><b>✅ Answer: Brute force — plain Python</b></summary>

**Idea:** Call strip and lower on each string.

```python
def tidy_names(data):
    return [n.strip().lower() for n in data["name"]]
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: Optimal — the library way</b></summary>

**Idea:** The `.str` accessor applies string methods to the whole column: `.str.strip().str.lower()`.

```python
def tidy_names(data):
    return pd.DataFrame(data)["name"].str.strip().str.lower().tolist()
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Call strip and lower on each string. That costs O(n) time and O(n) space."

"I can do better: The `.str` accessor applies string methods to the whole column: `.str.strip().str.lower()`. That's O(n) time and O(n) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/data-science/#/q/pandas-cleaning/tidy-up-text)

<a id="q7"></a>
### Q7. Month From a Date

🟢 Warm-up · *to\_datetime · .dt*

Dates are text like '2024-03-15'. Return the month number of each.

**Examples**

- `data = {'date': ['2024-03-15', '2023-12-01']}` → `[3, 12]`

In [ ]:
def months(data):
    """
    Dates are text like '2024-03-15'. Return the month number of each.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(months, [
    (({'date': ['2024-03-15', '2023-12-01']},), [3, 12]),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Split the text on '-' and read the middle part.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

`pd.to_datetime(s).dt.month` — real dates understand any format pandas can parse.

</details>

<details>
<summary><b>✅ Answer: Brute force — plain Python</b></summary>

**Idea:** Split the text on '-' and read the middle part.

```python
def months(data):
    return [int(d.split("-")[1]) for d in data["date"]]
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: Optimal — the library way</b></summary>

**Idea:** `pd.to_datetime(s).dt.month` — real dates understand any format pandas can parse.

```python
def months(data):
    return pd.to_datetime(pd.DataFrame(data)["date"]).dt.month.tolist()
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Split the text on '-' and read the middle part. That costs O(n) time and O(n) space."

"I can do better: `pd.to_datetime(s).dt.month` — real dates understand any format pandas can parse. That's O(n) time and O(n) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/data-science/#/q/pandas-cleaning/month-from-a-date)

<a id="q8"></a>
### Q8. Recode Categories

🟢 Warm-up · *map with a dict*

Replace short codes with full names using the mapping; unknown codes become 'other'.

**Examples**

- `data = {'code': ['M', 'F', 'X', 'M']}, mapping = {'M': 'male', 'F': 'female'}` → `['male', 'female', 'other', 'male']`

In [ ]:
def recode(data, mapping):
    """
    Replace short codes with full names using the mapping; unknown codes become
    'other'.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(recode, [
    (({'code': ['M', 'F', 'X', 'M']}, {'M': 'male', 'F': 'female'}), ['male', 'female', 'other', 'male']),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Loop with `mapping.get(code, 'other')`.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

`s.map(mapping).fillna('other')`.

</details>

<details>
<summary><b>✅ Answer: Brute force — plain Python</b></summary>

**Idea:** Loop with `mapping.get(code, 'other')`.

```python
def recode(data, mapping):
    return [mapping.get(c, "other") for c in data["code"]]
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: Optimal — the library way</b></summary>

**Idea:** `s.map(mapping).fillna('other')`.

```python
def recode(data, mapping):
    return pd.DataFrame(data)["code"].map(mapping).fillna("other").tolist()
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Loop with `mapping.get(code, 'other')`. That costs O(n) time and O(n) space."

"I can do better: `s.map(mapping).fillna('other')`. That's O(n) time and O(n) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/data-science/#/q/pandas-cleaning/recode-categories)

<a id="q9"></a>
### Q9. Cap Outliers

🟢 Warm-up · *clip*

Limit incomes to the range [lo, hi] so extreme values don't distort averages.

**Examples**

- `data = {'income': [20, 35, 900, -5]}, lo = 0, hi = 100` → `[20, 35, 100, 0]`

In [ ]:
def cap(data, lo, hi):
    """
    Limit incomes to the range [lo, hi] so extreme values don't distort
    averages.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(cap, [
    (({'income': [20, 35, 900, -5]}, 0, 100), [20, 35, 100, 0]),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

min/max in a loop.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

`s.clip(lo, hi)`.

</details>

<details>
<summary><b>✅ Answer: Brute force — plain Python</b></summary>

**Idea:** min/max in a loop.

```python
def cap(data, lo, hi):
    return [max(lo, min(v, hi)) for v in data["income"]]
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: Optimal — the library way</b></summary>

**Idea:** `s.clip(lo, hi)`.

```python
def cap(data, lo, hi):
    return pd.DataFrame(data)["income"].clip(lo, hi).tolist()
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: min/max in a loop. That costs O(n) time and O(n) space."

"I can do better: `s.clip(lo, hi)`. That's O(n) time and O(n) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/data-science/#/q/pandas-cleaning/cap-outliers)

<a id="q10"></a>
### Q10. Clean Column Names

🟢 Warm-up · *Renaming with str methods*

Turn column names like ' First Name ' into 'first\_name' (strip, lower, spaces → \_).

**Examples**

- `data = {' First Name ': [1], 'AGE': [2], 'Home City': [3]}` → `['first_name', 'age', 'home_city']`

In [ ]:
def clean_columns(data):
    """
    Turn column names like ' First Name ' into 'first_name' (strip, lower,
    spaces → _).
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(clean_columns, [
    (({' First Name ': [1], 'AGE': [2], 'Home City': [3]},), ['first_name', 'age', 'home_city']),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Loop over the names.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

`df.columns.str.strip().str.lower().str.replace(' ', '_')`.

</details>

<details>
<summary><b>✅ Answer: Brute force — plain Python</b></summary>

**Idea:** Loop over the names.

```python
def clean_columns(data):
    return [c.strip().lower().replace(" ", "_") for c in data]
```

**Cost:** O(cols) time, O(cols) space

</details>

<details>
<summary><b>✅ Answer: Optimal — the library way</b></summary>

**Idea:** `df.columns.str.strip().str.lower().str.replace(' ', '_')`.

```python
def clean_columns(data):
    df = pd.DataFrame(data)
    df.columns = df.columns.str.strip().str.lower().str.replace(" ", "_")
    return df.columns.tolist()
```

**Cost:** O(cols) time, O(cols) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Loop over the names. That costs O(cols) time and O(cols) space."

"I can do better: `df.columns.str.strip().str.lower().str.replace(' ', '_')`. That's O(cols) time and O(cols) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/data-science/#/q/pandas-cleaning/clean-column-names)

<a id="q11"></a>
### Q11. Carry the Last Reading Forward

🟢 Warm-up · *ffill*

A sensor sometimes misses a reading (None). Replace each gap with the last known reading.

**Examples**

- `data = {'temp': [21, None, None, 24, None]}` → `[21.0, 21.0, 21.0, 24.0, 24.0]`

In [ ]:
def carry_forward(data):
    """
    A sensor sometimes misses a reading (None). Replace each gap with the last
    known reading.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(carry_forward, [
    (({'temp': [21, None, None, 24, None]},), [21.0, 21.0, 21.0, 24.0, 24.0]),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Remember the last real value in a loop.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

`s.ffill()` (forward fill).

</details>

<details>
<summary><b>✅ Answer: Brute force — plain Python</b></summary>

**Idea:** Remember the last real value in a loop.

```python
def carry_forward(data):
    out, last = [], None
    for v in data["temp"]:
        if v is not None:
            last = v
        out.append(float(last) if last is not None else None)
    return out
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: Optimal — the library way</b></summary>

**Idea:** `s.ffill()` (forward fill).

```python
def carry_forward(data):
    return pd.DataFrame(data)["temp"].ffill().tolist()
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Remember the last real value in a loop. That costs O(n) time and O(n) space."

"I can do better: `s.ffill()` (forward fill). That's O(n) time and O(n) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/data-science/#/q/pandas-cleaning/carry-the-last-reading-forward)

<a id="q12"></a>
### Q12. Text to Numbers

🟡 Interview standard · *pd.to\_numeric(errors='coerce')*

Convert a column of text to numbers; anything that isn't a number becomes missing (None).

**Examples**

- `data = {'price': ['10', 'abc', '2.5', '']}` → `[10.0, None, 2.5, None]`

In [ ]:
def to_numbers(data):
    """
    Convert a column of text to numbers; anything that isn't a number becomes
    missing (None).
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(to_numbers, [
    (({'price': ['10', 'abc', '2.5', '']},), [10.0, None, 2.5, None]),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

try float(...) except ValueError in a loop.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

`pd.to_numeric(s, errors='coerce')` turns bad values into NaN.

</details>

<details>
<summary><b>✅ Answer: Brute force — plain Python</b></summary>

**Idea:** try float(...) except ValueError in a loop.

```python
def to_numbers(data):
    out = []
    for v in data["price"]:
        try:
            out.append(float(v))
        except ValueError:
            out.append(None)
    return out
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: Optimal — the library way</b></summary>

**Idea:** `pd.to_numeric(s, errors='coerce')` turns bad values into NaN.

```python
def to_numbers(data):
    return pd.to_numeric(pd.DataFrame(data)["price"], errors="coerce").tolist()
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: try float(...) except ValueError in a loop. That costs O(n) time and O(n) space."

"I can do better: `pd.to_numeric(s, errors='coerce')` turns bad values into NaN. That's O(n) time and O(n) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/data-science/#/q/pandas-cleaning/text-to-numbers)

<a id="q13"></a>
### Q13. Prices From Messy Text

🟡 Interview standard · *str.replace with regex*

Turn strings like '₹1,200' or 'Rs 75' into numbers by removing everything except digits.

**Examples**

- `data = {'price': ['₹1,200', 'Rs 75', '300']}` → `[1200, 75, 300]`

In [ ]:
def clean_prices(data):
    """
    Turn strings like '₹1,200' or 'Rs 75' into numbers by removing everything
    except digits.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(clean_prices, [
    (({'price': ['₹1,200', 'Rs 75', '300']},), [1200, 75, 300]),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Keep only digit characters in a loop, then int().

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

`.str.replace(r'\D', '', regex=True)` deletes every non-digit, then `astype(int)`.

</details>

<details>
<summary><b>✅ Answer: Brute force — plain Python</b></summary>

**Idea:** Keep only digit characters in a loop, then int().

```python
def clean_prices(data):
    out = []
    for text in data["price"]:
        digits = "".join(ch for ch in text if ch.isdigit())
        out.append(int(digits))
    return out
```

**Cost:** O(total chars) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: Optimal — the library way</b></summary>

**Idea:** `.str.replace(r'\D', '', regex=True)` deletes every non-digit, then `astype(int)`.

```python
def clean_prices(data):
    s = pd.DataFrame(data)["price"]
    return s.str.replace(r"\D", "", regex=True).astype(int).tolist()
```

**Cost:** O(total chars) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Keep only digit characters in a loop, then int(). That costs O(total chars) time and O(n) space."

"I can do better: `.str.replace(r'\D', '', regex=True)` deletes every non-digit, then `astype(int)`. That's O(total chars) time and O(n) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/data-science/#/q/pandas-cleaning/prices-from-messy-text)

<a id="q14"></a>
### Q14. Split Full Names

🟡 Interview standard · *str.split(expand=True)*

Split 'First Last' names into two columns; return {'first': […], 'last': […]}.

**Examples**

- `data = {'name': ['Ada Lovelace', 'Alan Turing']}` → `{'first': ['Ada', 'Alan'], 'last': ['Lovelace', 'Turing']}`

In [ ]:
def split_names(data):
    """
    Split 'First Last' names into two columns; return {'first': […], 'last':
    […]}.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(split_names, [
    (({'name': ['Ada Lovelace', 'Alan Turing']},), {'first': ['Ada', 'Alan'], 'last': ['Lovelace', 'Turing']}),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Split each string and append the parts to two lists.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

`s.str.split(' ', n=1, expand=True)` returns a DataFrame of the parts.

</details>

<details>
<summary><b>✅ Answer: Brute force — plain Python</b></summary>

**Idea:** Split each string and append the parts to two lists.

```python
def split_names(data):
    first, last = [], []
    for full in data["name"]:
        f, l = full.split(" ", 1)
        first.append(f)
        last.append(l)
    return {"first": first, "last": last}
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: Optimal — the library way</b></summary>

**Idea:** `s.str.split(' ', n=1, expand=True)` returns a DataFrame of the parts.

```python
def split_names(data):
    parts = pd.DataFrame(data)["name"].str.split(" ", n=1, expand=True)
    parts.columns = ["first", "last"]
    return parts
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Split each string and append the parts to two lists. That costs O(n) time and O(n) space."

"I can do better: `s.str.split(' ', n=1, expand=True)` returns a DataFrame of the parts. That's O(n) time and O(n) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/data-science/#/q/pandas-cleaning/split-full-names)

<a id="q15"></a>
### Q15. Days Between Two Dates

🟡 Interview standard · *Date arithmetic*

Return the number of days from `start` to `end` for each row.

**Examples**

- `data = {'start': ['2024-01-01', '2024-02-27'], 'end': ['2024-01-31', '2024-03-01']}` → `[30, 3]`

In [ ]:
def days_between(data):
    """
    Return the number of days from start to end for each row.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(days_between, [
    (({'start': ['2024-01-01', '2024-02-27'], 'end': ['2024-01-31', '2024-03-01']},), [30, 3]),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Use datetime.date.fromisoformat and subtract in a loop.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Subtract two datetime columns and read `.dt.days`.

</details>

<details>
<summary><b>✅ Answer: Brute force — plain Python</b></summary>

**Idea:** Use datetime.date.fromisoformat and subtract in a loop.

```python
def days_between(data):
    from datetime import date
    out = []
    for s, e in zip(data["start"], data["end"]):
        out.append((date.fromisoformat(e) - date.fromisoformat(s)).days)
    return out
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: Optimal — the library way</b></summary>

**Idea:** Subtract two datetime columns and read `.dt.days`.

```python
def days_between(data):
    df = pd.DataFrame(data)
    return (pd.to_datetime(df["end"]) - pd.to_datetime(df["start"])).dt.days.tolist()
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Use datetime.date.fromisoformat and subtract in a loop. That costs O(n) time and O(n) space."

"I can do better: Subtract two datetime columns and read `.dt.days`. That's O(n) time and O(n) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/data-science/#/q/pandas-cleaning/days-between-two-dates)

<a id="q16"></a>
### Q16. Age Groups

🟡 Interview standard · *pd.cut*

Label ages: 0–17 'child', 18–59 'adult', 60+ 'senior'.

**Examples**

- `data = {'age': [5, 18, 45, 60, 82]}` → `['child', 'adult', 'adult', 'senior', 'senior']`

In [ ]:
def age_groups(data):
    """
    Label ages: 0–17 'child', 18–59 'adult', 60+ 'senior'.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(age_groups, [
    (({'age': [5, 18, 45, 60, 82]},), ['child', 'adult', 'adult', 'senior', 'senior']),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

if/elif in a loop.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

`pd.cut(age, bins=[-1, 17, 59, 200], labels=[…])` — bins are (left, right].

</details>

<details>
<summary><b>✅ Answer: Brute force — plain Python</b></summary>

**Idea:** if/elif in a loop.

```python
def age_groups(data):
    out = []
    for a in data["age"]:
        if a <= 17:
            out.append("child")
        elif a <= 59:
            out.append("adult")
        else:
            out.append("senior")
    return out
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: Optimal — the library way</b></summary>

**Idea:** `pd.cut(age, bins=[-1, 17, 59, 200], labels=[…])` — bins are (left, right].

```python
def age_groups(data):
    groups = pd.cut(pd.DataFrame(data)["age"], bins=[-1, 17, 59, 200], labels=["child", "adult", "senior"])
    return groups.astype(str).tolist()
```

**Cost:** O(n log bins) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: if/elif in a loop. That costs O(n) time and O(n) space."

"I can do better: `pd.cut(age, bins=[-1, 17, 59, 200], labels=[…])` — bins are (left, right]. That's O(n log bins) time and O(n) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/data-science/#/q/pandas-cleaning/age-groups)

<a id="q17"></a>
### Q17. Fix Column Types

🟡 Interview standard · *astype*

Convert 'qty' to integers and 'price' to floats; return {column: dtype name}.

**Examples**

- `data = {'qty': ['1', '2'], 'price': ['9.5', '3']}` → `{'qty': 'int64', 'price': 'float64'}`

In [ ]:
def fixed_types(data):
    """
    Convert 'qty' to integers and 'price' to floats; return {column: dtype
    name}.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(fixed_types, [
    (({'qty': ['1', '2'], 'price': ['9.5', '3']},), {'qty': 'int64', 'price': 'float64'}),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Convert every value with int() / float() and report the Python types.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

`df.astype({'qty': 'int64', 'price': 'float64'})`, then read `df.dtypes`.

</details>

<details>
<summary><b>✅ Answer: Brute force — plain Python</b></summary>

**Idea:** Convert every value with int() / float() and report the Python types.

```python
def fixed_types(data):
    qty = [int(v) for v in data["qty"]]
    price = [float(v) for v in data["price"]]
    return {"qty": "int64" if all(isinstance(v, int) for v in qty) else "object",
            "price": "float64" if all(isinstance(v, float) for v in price) else "object"}
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: Optimal — the library way</b></summary>

**Idea:** `df.astype({'qty': 'int64', 'price': 'float64'})`, then read `df.dtypes`.

```python
def fixed_types(data):
    df = pd.DataFrame(data).astype({"qty": "int64", "price": "float64"})
    return {c: str(t) for c, t in df.dtypes.items()}
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Convert every value with int() / float() and report the Python types. That costs O(n) time and O(n) space."

"I can do better: `df.astype({'qty': 'int64', 'price': 'float64'})`, then read `df.dtypes`. That's O(n) time and O(n) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/data-science/#/q/pandas-cleaning/fix-column-types)

<a id="q18"></a>
### Q18. Find Invalid Emails

🟡 Interview standard · *str.contains with regex*

Return the emails that don't look like 'something@something.something'.

**Examples**

- `data = {'email': ['a@b.com', 'bad.mail', 'x@y', 'z@q.org']}` → `['bad.mail', 'x@y']`

In [ ]:
def invalid_emails(data):
    """
    Return the emails that don't look like 'something@something.something'.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(invalid_emails, [
    (({'email': ['a@b.com', 'bad.mail', 'x@y', 'z@q.org']},), ['bad.mail', 'x@y']),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Check with string methods: one '@' and a '.' after it.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

A regular expression with `.str.match` tests every row at once.

</details>

<details>
<summary><b>✅ Answer: Brute force — plain Python</b></summary>

**Idea:** Check with string methods: one '@' and a '.' after it.

```python
def invalid_emails(data):
    bad = []
    for e in data["email"]:
        if e.count("@") != 1:
            bad.append(e)
            continue
        user, domain = e.split("@")
        if not user or "." not in domain or domain.startswith(".") or domain.endswith("."):
            bad.append(e)
    return bad
```

**Cost:** O(total chars) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: Optimal — the library way</b></summary>

**Idea:** A regular expression with `.str.match` tests every row at once.

```python
def invalid_emails(data):
    s = pd.DataFrame(data)["email"]
    ok = s.str.match(r"^[^@\s]+@[^@\s]+\.[^@\s]+$")
    return s[~ok].tolist()
```

**Cost:** O(total chars) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Check with string methods: one '@' and a '.' after it. That costs O(total chars) time and O(n) space."

"I can do better: A regular expression with `.str.match` tests every row at once. That's O(total chars) time and O(n) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/data-science/#/q/pandas-cleaning/find-invalid-emails)

<a id="q19"></a>
### Q19. Fill Gaps by Interpolation

🟡 Interview standard · *interpolate*

Fill missing values in a series by drawing a straight line between the known neighbours.

**Examples**

- `data = {'v': [10, None, 30, None, None, 60]}` → `[10.0, 20.0, 30.0, 40.0, 50.0, 60.0]`

In [ ]:
def interpolate(data):
    """
    Fill missing values in a series by drawing a straight line between the known
    neighbours.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(interpolate, [
    (({'v': [10, None, 30, None, None, 60]},), [10.0, 20.0, 30.0, 40.0, 50.0, 60.0]),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

For each gap, find the known values on both sides and divide the difference evenly.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

`s.interpolate()` (linear by default).

</details>

<details>
<summary><b>✅ Answer: Brute force — plain Python</b></summary>

**Idea:** For each gap, find the known values on both sides and divide the difference evenly.

```python
def interpolate(data):
    v = list(data["v"])
    known = [i for i, x in enumerate(v) if x is not None]
    for a, b in zip(known, known[1:]):
        step = (v[b] - v[a]) / (b - a)
        for i in range(a + 1, b):
            v[i] = v[a] + step * (i - a)
    return [float(x) for x in v]
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: Optimal — the library way</b></summary>

**Idea:** `s.interpolate()` (linear by default).

```python
def interpolate(data):
    return pd.DataFrame(data)["v"].interpolate().tolist()
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: For each gap, find the known values on both sides and divide the difference evenly. That costs O(n) time and O(n) space."

"I can do better: `s.interpolate()` (linear by default). That's O(n) time and O(n) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/data-science/#/q/pandas-cleaning/fill-gaps-by-interpolation)

<a id="q20"></a>
### Q20. Drop Mostly-Empty Columns

🟡 Interview standard · *dropna(axis=1, thresh=…)*

Remove columns where more than half the values are missing; return the remaining column names.

**Examples**

- `data = {'a': [1, 2, 3, 4], 'b': [None, None, None, 1], 'c': [1, None, 2, None]}` → `['a', 'c']`

In [ ]:
def keep_columns(data):
    """
    Remove columns where more than half the values are missing; return the
    remaining column names.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(keep_columns, [
    (({'a': [1, 2, 3, 4], 'b': [None, None, None, 1], 'c': [1, None, 2, None]},), ['a', 'c']),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Count missing values per column and keep the column if missing ≤ half.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

`df.dropna(axis=1, thresh=…)` keeps columns with at least thresh non-missing values.

</details>

<details>
<summary><b>✅ Answer: Brute force — plain Python</b></summary>

**Idea:** Count missing values per column and keep the column if missing ≤ half.

```python
def keep_columns(data):
    out = []
    for col, values in data.items():
        missing = sum(v is None for v in values)
        if missing <= len(values) / 2:
            out.append(col)
    return out
```

**Cost:** O(cells) time, O(cols) space

</details>

<details>
<summary><b>✅ Answer: Optimal — the library way</b></summary>

**Idea:** `df.dropna(axis=1, thresh=…)` keeps columns with at least thresh non-missing values.

```python
def keep_columns(data):
    df = pd.DataFrame(data)
    return df.dropna(axis=1, thresh=(len(df) + 1) // 2).columns.tolist()
```

**Cost:** O(cells) time, O(cells) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Count missing values per column and keep the column if missing ≤ half. That costs O(cells) time and O(cols) space."

"I can do better: `df.dropna(axis=1, thresh=…)` keeps columns with at least thresh non-missing values. That's O(cells) time and O(cells) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/data-science/#/q/pandas-cleaning/drop-mostly-empty-columns)

---
# Part 4: Interview questions, out loud

Cover the answer, **say yours out loud**, then open it and compare. Aim for two or three clear sentences, then an example.

<details>
<summary><b>1. dropna or fillna?</b></summary>

Drop when few rows are affected and missingness is random; fill when you can't afford to lose rows or the gap has a sensible value. Always record the choice, and consider an indicator column.

</details>

<details>
<summary><b>2. Why fill with the median rather than the mean?</b></summary>

The median isn't pulled by outliers, so it's a safer typical value for skewed data such as incomes or prices.

</details>

<details>
<summary><b>3. What does errors='coerce' do?</b></summary>

Values that can't be converted become NaN (or NaT for dates) instead of raising an error, so you can count and handle them afterwards.

</details>

<details>
<summary><b>4. How do you find duplicate rows?</b></summary>

`df.duplicated()` marks repeats (all but the first by default); `df.duplicated().sum()` counts them; `drop_duplicates(subset=[...])` removes them based on chosen columns.

</details>

<details>
<summary><b>5. ffill vs interpolate?</b></summary>

ffill copies the last known value forward (step-like). interpolate draws a line between known neighbours (smooth). Both need the data in a meaningful order.

</details>

<details>
<summary><b>6. What is the category dtype for?</b></summary>

Columns with few distinct repeated values (city, grade). Values are stored as small integer codes plus a lookup table, saving memory and speeding up groupby.

</details>


---
# Check yourself before moving on

You're ready for the next notebook when you can tick these **without looking**:

- [ ] I can explain: Why clean data?
- [ ] I can explain: How missing values really behave
- [ ] I can explain: Missing values
- [ ] I can explain: Types and duplicates
- [ ] I can explain: Cleaning text with .str
- [ ] I can explain: Choosing how to fill gaps, worked
- [ ] I can explain: Outliers: find them, then decide
- [ ] I can explain: Dates, categories and outliers
- [ ] I can explain: A complete cleaning walk-through

**Revision drill.** Come back in a few days, set a 25-minute timer per question, and solve these from a blank cell:

- [ ] [Q16](#q16) Age Groups
- [ ] [Q17](#q17) Fix Column Types
- [ ] [Q18](#q18) Find Invalid Emails
- [ ] [Q19](#q19) Fill Gaps by Interpolation
- [ ] [Q20](#q20) Drop Mostly-Empty Columns

Spaced repetition beats re-reading: redoing a question after you've partly forgotten it is what makes it stick.